In [12]:
import re
import math
import random
from pathlib import Path
from collections import Counter, defaultdict

def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

sherlock_text = Path("data/sherlock.txt").read_text(encoding="utf-8")
pride_text = Path("data/pride.txt").read_text(encoding="utf-8")

all_sents = to_sentences(sherlock_text) + to_sentences(pride_text)
random.Random(42).shuffle(all_sents)

split_idx = int(0.85 * len(all_sents))
train_sents = all_sents[:split_idx]
test_sents = all_sents[split_idx:]
train_half = train_sents[: len(train_sents) // 2]

print(f"Total sentences : {len(all_sents):,}")
print(f"Train sentences : {len(train_sents):,} (Half slice: {len(train_half):,})")
print(f"Test sentences  : {len(test_sents):,}")

Total sentences : 14,648
Train sentences : 12,450 (Half slice: 6,225)
Test sentences  : 2,198


In [13]:
class NgramLM:
    def __init__(self, n=2, k=0.5, min_freq=2, lambdas=None):
        self.n = n
        self.k = k
        self.min_freq = min_freq
        self.lambdas = lambdas

    def _replace_unk(self, tokens):
        return [t if t in self.V else "<unk>" for t in tokens]

    def fit(self, sents):
        raw_counts = Counter(t for s in sents for t in to_tokens(s))
        self.V = {w for w, c in raw_counts.items() if c >= self.min_freq} | {"</s>", "<unk>"}
        self.tables = {}
        for order in range(1, self.n + 1):
            foll = defaultdict(Counter)
            for s in sents:
                toks = ["<s>"] * (order - 1) + self._replace_unk(to_tokens(s)) + ["</s>"]
                for i in range(len(toks) - order + 1):
                    ctx = tuple(toks[i : i + order - 1])
                    nxt = toks[i + order - 1]
                    foll[ctx][nxt] += 1
            self.tables[order] = foll
        return self

    def _order_prob(self, order, ctx, w):
        w = w if w in self.V else "<unk>"
        ctx = tuple(t if t in self.V or t == "<s>" else "<unk>" for t in ctx)
        c = self.tables[order].get(ctx, Counter())
        total = sum(c.values())
        return (c[w] + self.k) / (total + self.k * len(self.V))

    def prob(self, ctx, w):
        if not self.lambdas:
            return self._order_prob(self.n, ctx, w)
        p = 0.0
        for order, lam in enumerate(self.lambdas, start=1):
            sub_ctx = ctx[-(order - 1):] if order > 1 else ()
            p += lam * self._order_prob(order, sub_ctx, w)
        return p

    def suggest_next(self, text, topk=3):
        toks = self._replace_unk(to_tokens(text))
        for order in range(self.n, 0, -1):
            ctx = tuple((["<s>"] * (order - 1) + toks)[-(order - 1):]) if order > 1 else ()
            c = self.tables[order].get(ctx, Counter())
            words = [w for w, _ in c.most_common() if w not in ("</s>", "<unk>")]
            if words:
                return words[:topk]
        return ["(no idea)"]

    def generate(self, seed=0, max_len=30):
        rng = random.Random(seed)
        ctx = ("<s>",) * (self.n - 1)
        out = []
        for _ in range(max_len):
            c = None
            for order in range(self.n, 0, -1):
                sub_ctx = ctx[-(order - 1):] if order > 1 else ()
                cand = {w: cnt for w, cnt in self.tables[order].get(sub_ctx, {}).items() if w != "<unk>"}
                if cand:
                    c = cand
                    break
            if not c:
                break
            words, wts = zip(*c.items())
            nxt = rng.choices(words, weights=wts)[0]
            if nxt == "</s>":
                break
            out.append(nxt)
            if self.n > 1:
                ctx = (ctx + (nxt,))[-(self.n - 1):]
        return " ".join(out)

    def perplexity(self, sents):
        log_sum, N = 0.0, 0
        for s in sents:
            toks = ["<s>"] * (self.n - 1) + to_tokens(s) + ["</s>"]
            for i in range(len(toks) - self.n + 1):
                ctx = tuple(toks[i : i + self.n - 1])
                w = toks[i + self.n - 1]
                log_sum += math.log(self.prob(ctx, w))
                N += 1
        return math.exp(-log_sum / N)

In [14]:
lm_tri = NgramLM(n=3, k=0.1, min_freq=2).fit(train_sents)
print(f"Vocabulary size |V| (min_freq=2): {len(lm_tri.V):,}")

ctx = ("in", "the")
counts = lm_tri.tables[3][ctx]
total_ctx = sum(counts.values())

print(f"\nContext: {ctx} (appeared {total_ctx} times in training)\n")
print(f"{'Next Word':<15} {'Raw Count':>10} {'MLE P(w|ctx)':>15} {'Smoothed P(w|ctx)':>20}")
print("-" * 64)

for word, cnt in counts.most_common(8):
    mle = cnt / total_ctx
    smoothed = lm_tri._order_prob(3, ctx, word)
    print(f"{word:<15} {cnt:>10} {mle:>15.4f} {smoothed:>20.6f}")

unseen = "spaceship"
print(f"{unseen + ' (unseen)':<15} {0:>10} {0.0:>15.4f} {lm_tri._order_prob(3, ctx, unseen):>20.6f}")

Vocabulary size |V| (min_freq=2): 6,178

Context: ('in', 'the') (appeared 788 times in training)

Next Word        Raw Count    MLE P(w|ctx)    Smoothed P(w|ctx)
----------------------------------------------------------------
world                   31          0.0393             0.022123
<unk>                   29          0.0368             0.020700
morning                 23          0.0292             0.016432
room                    17          0.0216             0.012164
evening                 17          0.0216             0.012164
course                  14          0.0178             0.010030
house                   13          0.0165             0.009319
country                 13          0.0165             0.009319
spaceship (unseen)          0          0.0000             0.020700


In [15]:
lm_bi = NgramLM(n=2, k=0.1, min_freq=2).fit(train_sents)

print("=== BIGRAM (n=2) vs. TRIGRAM (n=3) GENERATION ===\n")
for seed in range(1, 6):
    print(f"Seed {seed}:")
    print(f"  [n=2] {lm_bi.generate(seed=seed, max_len=25)}")
    print(f"  [n=3] {lm_tri.generate(seed=seed, max_len=25)}\n")

=== BIGRAM (n=2) vs. TRIGRAM (n=3) GENERATION ===

Seed 1:
  [n=2] what it may be happy promise you there was a baboon
  [n=3] what did she accepted him

Seed 2:
  [n=2] iii
  [n=3] iii

Seed 3:
  [n=2] i would give me as to hear a cheetah and high respect he did you must allow in an indirect or feel how maddening it
  [n=3] i will do it without abhorrence

Seed 4:
  [n=2] i had long to the window enter through a heavy brown hands
  [n=3] i am glad of it and after receiving her brother is the cause of all things

Seed 5:
  [n=2] but he have long before anybody bringing forward neither could ha
  [n=3] but my father can you contrive to write again very soon



In [16]:
print("=== HELD-OUT PERPLEXITY GRID (Add-k Smoothing) ===\n")
k_values = [0.001, 0.01, 0.1, 0.5, 1.0]

header = f"{'Model':<14} | " + " | ".join(f"k={k:<6}" for k in k_values)
print(header)
print("-" * len(header))

for n in [1, 2, 3]:
    row = []
    for k in k_values:
        model = NgramLM(n=n, k=k, min_freq=2).fit(train_sents)
        ppl = model.perplexity(test_sents)
        row.append(f"{ppl:8.1f}")
    print(f"n={n:<12} | " + " | ".join(row))

tri_half = NgramLM(n=3, k=0.01, min_freq=2).fit(train_half)
tri_full = NgramLM(n=3, k=0.01, min_freq=2).fit(train_sents)
tri_interp = NgramLM(n=3, k=0.01, min_freq=2, lambdas=[0.1, 0.3, 0.6]).fit(train_sents)

print("\n=== DOES MORE DATA / INTERPOLATION FIX THE TRIGRAM? ===")
print(f"Pure Trigram (50% data, k=0.01)         : {tri_half.perplexity(test_sents):.1f}")
print(f"Pure Trigram (100% data, k=0.01)        : {tri_full.perplexity(test_sents):.1f}")
print(f"Interpolated Trigram (100% data, k=0.01): {tri_interp.perplexity(test_sents):.1f}")

=== HELD-OUT PERPLEXITY GRID (Add-k Smoothing) ===

Model          | k=0.001  | k=0.01   | k=0.1    | k=0.5    | k=1.0   
---------------------------------------------------------------------
n=1            |    421.2 |    421.2 |    421.3 |    422.1 |    423.4
n=2            |    302.4 |    254.1 |    339.8 |    584.0 |    792.3
n=3            |   1123.2 |   1081.6 |   1649.1 |   2501.2 |   2972.2

=== DOES MORE DATA / INTERPOLATION FIX THE TRIGRAM? ===
Pure Trigram (50% data, k=0.01)         : 1014.0
Pure Trigram (100% data, k=0.01)        : 1081.6
Interpolated Trigram (100% data, k=0.01): 240.4


In [17]:
prompts = [
    "mr darcy",
    "mr darcy was",
    "sherlock holmes",
    "it is a truth",
    "in the",
    "she could not",
]

print("=== AUTOCOMPLETE PREDICTIONS (Interpolated Trigram) ===\n")
for p in prompts:
    suggestions = tri_interp.suggest_next(p, topk=5)
    print(f"{p!r:20s} -> {suggestions}")

=== AUTOCOMPLETE PREDICTIONS (Interpolated Trigram) ===

'mr darcy'           -> ['s', 'was', 'and', 'had', 'is']
'mr darcy was'       -> ['expected', 'not', 'to', 'one', 'injury']
'sherlock holmes'    -> ['had', 'was', 'sat', 'returned', 'by']
'it is a truth'      -> ['universally']
'in the'             -> ['world', 'morning', 'room', 'evening', 'course']
'she could not'      -> ['be', 'help', 'have', 'but', 'possibly']


## Findings & Takeaways

1. **Counts to Probabilities & Smoothing:** Unseen n-grams receive non-zero probability via add-$k$ smoothing, preventing $-\infty$ log-likelihoods during evaluation while preserving the relative ranking of frequent continuations.
2. **Bigram vs. Trigram Generation:** On the combined *Sherlock Holmes* + *Pride and Prejudice* corpus (~13k sentences), trigrams (`n=3`) produce noticeably more coherent phrasing than bigrams (`n=2`) because a 2-word context locks in local grammar and character names.
3. **Does More Data Fix the Trigram? (Perplexity Results):**
   - **Pure Trigram (50% data, `k=0.01`):** `1014.0`
   - **Pure Trigram (100% data, `k=0.01`):** `1081.6`
   - **Interpolated Trigram (100% data, `k=0.01`):** `240.4`

   Simply adding more data does **not** fix a pure add-$k$ trigram: perplexity actually edged up from `1014.0` to `1081.6` because the larger corpus expanded the active vocabulary $|V|$ (`min_freq=2`), increasing the $k \cdot |V|$ smoothing penalty across sparse, unseen 2-word contexts. Blending unigram, bigram, and trigram probabilities via linear interpolation (`lambdas=[0.1, 0.3, 0.6]`) resolves this context sparsity and slashes held-out perplexity to `240.4`.